### 테이블 나누기
- 대출번호/회원이름/회원전화/제목/저자/대출일 등..
- 위 처럼 저장했을 때의 문제
    - 회원이 책 빌릴 때마다 -> 회원이름/전화번호 데이터조회
    - 이때 전화번호 바뀌면 -> 모든 행 다 수정해야 하며 중복, 불일치 문제들 발생

- 해결 방법 : 테이블 나누기

- 회원 테이블, 책 테이블, 대출 테이블로 나눠서 정리할 수 있다!
- 회원 테이블 : 회원정보는 한번만 기록
- 책 테이블: 책 정보도 한번만
- 대출 테이블: 누가(회원정보) 어떤 책을(책번호) -> 연결 정보만 저장

### 세 테이블 관계 - ERD로 보기
- 1:N 관계 : 회원 1명은 대출을 여러 건 진행할 수 있음
- Foreign Key (FK) : 이 열은 저쪽 테이블의 기본키를 참조한다고 명시한다!
    - 여기서 기본 키란 기본 테이블의 카테고리 등등

In [1]:
import sqlite3
import pandas as pd

In [3]:
conn = sqlite3.connect("library_test.db")
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

In [4]:
# 1번째 테이블 만들기 - 회원정보

cur.execute("""
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    name TEXT,
    phone TEXT
)
""")

conn.commit()

In [5]:
# 2번째 테이블 만들기 - books

cur.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    author TEXT
)
""")

conn.commit()

In [6]:
# 3번째 테이블 만들기 - 연결정보

cur.execute("""
CREATE TABLE loans (
    loan_id INTEGER PRIMARY KEY,
    member_id INTEGER,
    book_id INTEGER,
    loan_date TEXT,
    FOREIGN KEY (member_id) REFERENCES members(member_id),
    FOREIGN KEY (book_id) REFERENCES books(book_id)
)
""")

conn.commit()

In [7]:
# 회원 등록 해보기
member_list = [
    (1, "이현근", "010-1111"),
    (2, "김수연", "010-2222"),
    (3, "송수림", "010-3333"),
    (4, "주승우", "010-4444"),
]

cur.executemany("INSERT INTO members VALUES (?, ?, ?)", member_list)
conn.commit()

In [8]:
# 책정보 등록 해보기
book_list = [
    (101, "오늘만 사는 법", "이현근2"),
    (102, "추석에 맛있는 거 먹는 법", "김수연3"),
    (103, "신기한 여행", "에드워드"),
    (104, "핸즈온 머신러닝", "오렐리안"),
]

cur.executemany("INSERT INTO books VALUES (?, ?, ?)", book_list)
conn.commit()

In [ ]:
# 대출기록 넣기 - 회원번호-책번호로만 연결
loan_list = [
    (1, 1, 101, "2026-08-01"),
    (2, 3, 104, "2026-08-02"),
    (3, 1, 102,"2026-08-02"),
    (4, 2, 103, "2026-08-04"),
    (5, 4, 102, "2026-08-05"),
    (6, 4, 104, "2026-08-08"),
]

cur.executemany("INSERT INTO loans VALUES (?, ?, ?, ?)", loan_list) # 들어갈 변수 중 2개는 외부키이므로 물음표 갯수에서 제외
conn.commit()

### JOIN

In [11]:
data = pd.read_sql("""
SELECT loans.loan_date AS 대출일,
        members.name AS 회원,
        books.title AS 책제목
FROM loans
JOIN members ON loans.member_id = members.member_id
JOIN books ON loans.book_id = books.book_id

""", conn)
data

,대출일,회원,책제목
0,2026-08-01,이현근,오늘만 사는 법
1,2026-08-02,송수림,핸즈온 머신러닝
2,2026-08-02,이현근,추석에 맛있는 거 먹는 법
3,2026-08-04,김수연,신기한 여행
4,2026-08-05,주승우,추석에 맛있는 거 먹는 법
5,2026-08-08,주승우,핸즈온 머신러닝


In [12]:
# 대출 건수가 많은 사람 조회하기
# groupby 회원별 대출수

data = pd.read_sql("""
SELECT members.name AS 회원,
        COUNT(*) AS 대출건수
FROM loans
JOIN members ON loans.member_id = members.member_id
GROUP BY members.name
""", conn)
data

,회원,대출건수
0,김수연,1
1,송수림,1
2,이현근,2
3,주승우,2
